# Echo Audio

Echo Audio is a multi-tap delay whose tap map is measured on a quantum circuit (Echo Core). A line of qubits runs forward, gets a nudge, then runs backwards; what fails to cancel becomes the echo pattern applied to your sound. Some echoes come back inverted, reversed or phase-rotated, which no room or tape delay can do. (The API names it Retrocausal Echo.)

## Setup

In [ ]:
import getpass
import os
import json

from moth_atlas import Atlas

ENGINE = "retrocausal-echo-staging"  # the engine this notebook runs: Atlas V2 (staging). Production: "retrocausal-echo-v1"

if not os.environ.get("MOTH_API_KEY"):
    os.environ["MOTH_API_KEY"] = getpass.getpass("Atlas API key")

client = Atlas()  # reads MOTH_API_KEY, and MOTH_API_BASE for which API to call

In [ ]:
import io
import wave

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Audio, display


def samples(wav):
    """A WAV file (path or bytes) as (float samples in [-1, 1], one column per channel; sample rate)."""
    with wave.open(wav if isinstance(wav, str) else io.BytesIO(wav)) as w:
        width, channels, rate = w.getsampwidth(), w.getnchannels(), w.getframerate()
        raw = np.frombuffer(w.readframes(w.getnframes()), np.uint8)
    if width == 3:  # 24-bit: pad each sample to 32 bits
        raw, width = np.pad(raw.reshape(-1, 3), ((0, 0), (1, 0))).reshape(-1), 4
    x = raw.view(f"<i{width}").astype(np.float32) / 2 ** (8 * width - 1)
    return x.reshape(-1, channels), rate


def listen(clips, play=True, stereo=False, rate=16000, height=1.1):
    """Plot and play {label: WAV path or bytes}, one row each. The players are resampled to `rate` (mono unless
    `stereo`) to keep this notebook small; the plots and downloads keep the full file."""
    fig, axes = plt.subplots(len(clips), 1, figsize=(9, height * len(clips)), sharex=True, squeeze=False)
    for ax, (label, wav) in zip(axes[:, 0], clips.items()):
        x, sr = samples(wav)
        t = np.arange(len(x)) / sr
        for c in range(x.shape[1]):
            ax.plot(t, x[:, c], lw=0.4, alpha=0.8)
        ax.set_ylim(-1, 1)
        ax.set_yticks([])
        ax.set_ylabel(label, rotation=0, ha="right", va="center")
    axes[-1, 0].set_xlabel("seconds")
    plt.show()
    for label, wav in clips.items() if play else ():
        x, sr = samples(wav)
        x = x if stereo else x.mean(axis=1, keepdims=True)
        t = np.arange(0, len(x) / sr, 1 / rate)
        y = np.stack([np.interp(t, np.arange(len(x)) / sr, x[:, c]) for c in range(x.shape[1])])
        print(label)
        display(Audio(y if stereo else y[0], rate=rate, normalize=False))

def output_asset(job, slot):
    """One output file of a finished job, as an Asset you can pass straight into another job's files."""
    out = next(o for o in job.outputs if o.slot == slot)
    return client.assets.get(out.output_asset_id)


def tap_map(job, ax=None):
    """Plot a job's echoes by time and stereo position, sized by level: red returns upright, blue inverted."""
    taps = json.loads(job.output_bytes("taps"))["extras"]
    s = taps["echo_summary"]
    show = ax is None
    ax = ax or plt.subplots(figsize=(9, 2))[1]
    for tap in taps["tap_map"]["taps"]:
        ax.scatter(tap["time_ms"], tap["pan"], s=10 + 300 * tap["level"], alpha=0.6,
                   color="C3" if tap["polarity"] > 0 else "C0")
    ax.set_ylim(-1.2, 1.2)
    ax.set_yticks([-1, 0, 1], ["L", "C", "R"])
    ax.set_xlabel("ms")
    ax.set_title(f"{s['regular']} regular · {s['inverted']} inverted · {s['erased']} erased", fontsize=9)
    if show:
        plt.show()

## Try it

`snare_crack.wav` is a short snare hit beside this notebook (from Moth's QEcho sample pack): short sounds make each echo easy to hear. The tap map shows every echo, by time and stereo position.

In [ ]:
job = client.jobs.run(
    engine=ENGINE,
    params={
        "theta_x": 0.94,
        "negative_mode": "invert",
        "mix": 0.6,
        "depth": 8,
        "master_ms": 640,
    },
    files={
        "audio": "snare_crack.wav",
    },
)
listen({"dry": "snare_crack.wav", "echo": job.output_bytes("result")}, stereo=True)
tap_map(job)

## Parameters

What to expect from each:

- `theta_x` (0 to π): how quantum. Low gives loud, regular echoes; high gives echoes that flip, fade or vanish.
- `negative_mode`: what an inverted echo does. `invert` flips its polarity (partly cancelling the dry sound), `reverse` plays its grains backwards (`grain_ms` sets their length), `phase` rotates it by the tap's phase (pair it with `theta_z` > 0).
- `mix` (0 to 1): dry to wet.
- `depth`, `master_ms`: how many echo slots, and how long the whole line is. Or set `bpm` and `division` to lock each slot to a note value (0.25 = sixteenths).
- `decay`, `feedback`, `feedback_source`, `max_regen`: extra roll-off per slot, and how much of the echo comes back round.
- `stereo_width`: how wide the echoes spread; 0 is mono.
- `theta_zz`, `theta_z`, `kick`, `kick_site`, `disorder`, `seed`, `n_sites`, `lattice`: the circuit itself, as in Echo Core.
- `emit`: `audio` (default) renders a WAV, `map` returns only the tap map for a sequencer or plugin. With no `audio` file, you get the effect's own impulse response (`ir_seconds` long).

Inputs: `audio` (WAV, MP3, OGG or FLAC; stereo is summed to mono) and `ir`, a previous measurement to re-render without measuring again. Outputs: `result` (the WAV), `ir` (the measurement) and `taps` (the rendered tap map).

In [ ]:
# client.engines.get(ENGINE)  # its description, params, input and output files, from the API

## How it works

**Measure.** The circuit is Echo Core's: $n$ qubits driven forward by $t$ layers of $R_{ZZ}(\theta_{zz})$ and $R_X(\theta_x)$, kicked by one Pauli on one qubit, then run backwards. For every site and depth it returns a number $F$: near 1 if the echo came back intact, negative if inverted, near 0 if scrambled away.

**Map.** Each $(\text{site}, t)$ becomes a tap: site → stereo position, depth → time ($t \cdot$ `master_ms` / `depth`), $|F|$ → level (times `decay` per step), and the sign of $F$ → what `negative_mode` does.

**Render.** Your audio passes through the taps as a multi-tap delay with optional feedback, and is mixed with the dry signal. Inverted taps partly cancel the dry sound, so the effect can sound hollow rather than wet.

The measurement and the rendering are separate: the `ir` output carries the measurement, so you can re-render it with any mix settings without running the circuit again.

## Experiments

Each experiment changes one parameter and keeps the others fixed, and runs one job per value.

### How quantum (`theta_x`)

Low: a regular, loud multi-tap. Higher: more echoes come back inverted (blue) or not at all.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(9, 5.5), sharex=True)
clips = {}
for ax, theta_x in zip(axes, (0.3, 0.94, 1.8)):
    print(f"theta_x={theta_x}")
    job = client.jobs.run(engine=ENGINE, params={"theta_x": theta_x}, files={"audio": "snare_crack.wav"}, quiet=True)
    tap_map(job, ax)
    ax.set_ylabel(f"theta_x {theta_x}")
    clips[f"theta_x {theta_x}"] = job.output_bytes("result")
plt.tight_layout()
plt.show()
listen(clips)

### Negative echoes

The same measurement rendered three ways, on a sustained Rhodes note: flipped, played backwards in grains, or phase-rotated (with a phase layer, `theta_z`, so the taps have a phase to rotate by).

In [ ]:
clips = {"dry": "rhodes_c4.wav"}
for mode, extra in (("invert", {}), ("reverse", {"grain_ms": 120}), ("phase", {"theta_z": 0.8})):
    params = {"negative_mode": mode, "theta_x": 1.2, **extra}
    print(params)
    job = client.jobs.run(engine=ENGINE, params=params, files={"audio": "rhodes_c4.wav"}, quiet=True)
    clips[mode] = job.output_bytes("result")
listen(clips)

### The effect on its own

With no audio file, the engine renders the effect's impulse response: what a single click would sound like through it. Load it into any convolution reverb.

In [ ]:
job = client.jobs.run(engine=ENGINE, params={"ir_seconds": 2, "theta_x": 1.2})
listen({"impulse response": job.output_bytes("result")})

### One measurement, many mixes

Keep the measurement from the first run (its `ir` output) and render it again with other mix settings. The circuit doesn't run again, so every take shares the same tap map.

In [ ]:
measured = client.jobs.run(engine=ENGINE, params={"theta_x": 1.2}, files={"audio": "snare_crack.wav"}, quiet=True)
ir = output_asset(measured, "ir")

clips = {}
for params in ({"mix": 0.3}, {"mix": 0.9}, {"mix": 0.6, "feedback": 0.6}, {"mix": 0.6, "bpm": 120, "division": 0.75}):
    print(params)
    job = client.jobs.run(engine=ENGINE, params=params, files={"audio": "snare_crack.wav", "ir": ir}, quiet=True)
    clips[", ".join(f"{k} {v}" for k, v in params.items())] = job.output_bytes("result")
listen(clips)

## Questions

**Why "retrocausal"?**

The echo pattern comes from running a circuit backwards in time after a disturbance. Nothing travels backwards in your audio; the name is for the measurement behind it.

**Why does it sometimes sound hollow?**

Inverted taps have the opposite sign to the dry signal, so they partly cancel it. Try `negative_mode` `reverse` or `phase`, or lower `theta_x` for fewer inversions.

**Can I use the tap map in a DAW or sequencer?**

Yes: `emit: "map"` skips rendering and returns only the tap list (time, level, pan, polarity per tap).

**Does it run on quantum hardware?**

On `aer` (default) it's simulated. Set `machine` to an IBM backend name to measure the map on a QPU (your account needs access); the rendering stays the same.

Sample sounds from Moth's QEcho sample pack (synthesised, no licence restrictions).